# silver_agents

First silver notebook that actually does something interesting. Agents come from two bronze sources (CSV and API), both covering the same 200 agents. This notebook reconciles them into one clean table and quarantines anything that doesn't pass validation.

The patterns here (UNION, source priority, reject quarantine) repeat in silver_policies and silver_claims. Get comfortable with them here on 200 rows before they show up on 10,000.

## Source priority

CSV is primary. When both sources have the same agent_id, CSV wins. API rows only make it into silver if there's no CSV equivalent for that agent_id.

Why CSV over API? The CSV simulates an IMS bulk export, the authoritative system of record. The API simulates a live feed, which is useful for catching new agents but not authoritative when both sources conflict.

## Outputs

- `silver_agents`: clean, deduplicated, validated
- `silver_agents_rejects`: rows that failed validation, with a reject_reason column saying why

Rows don't get silently dropped. If a row fails validation it goes to rejects. The row count math at the end should always satisfy: CSV rows + API-only rows = silver_agents + silver_agents_rejects.

## Validation

Three checks before a row makes it into silver_agents:

1. `tier` must be BRONZE, SILVER, GOLD, or PLATINUM
2. `commission_rate` must be between 0.0 and 0.20
3. `hire_date` must not be null

A row failing multiple checks gets one reject_reason string with all failures concatenated.

## What's NOT here

No FK validation. Agents don't FK out to anything; policies FK to agents, not the other way around. FK enforcement shows up in silver_policies.

No SCD2. Tier and commission_rate are the obvious candidates when an agent gets promoted. Worth building in a future pass but out of scope here.


In [2]:
from pyspark.sql.functions import lit, current_timestamp, col, when, concat_ws
from pyspark.sql.types import StringType
import uuid
from datetime import datetime

SILVER_BATCH_ID = f"silver_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

# Two bronze sources, same entities.
SOURCE_CSV = "bronze_citadel_mga.dbo.bronze_agents"
SOURCE_API = "bronze_citadel_mga.dbo.bronze_agents_api"

# Two silver outputs.
TARGET_TABLE  = "silver_agents"
REJECTS_TABLE = "silver_agents_rejects"

# Allowed tier values. Any row outside this set fails validation.
VALID_TIERS = {"BRONZE", "SILVER", "GOLD", "PLATINUM"}

print(f"Silver batch ID: {SILVER_BATCH_ID}")
print(f"CSV source:      {SOURCE_CSV}")
print(f"API source:      {SOURCE_API}")
print(f"Target:          {TARGET_TABLE}")
print(f"Rejects:         {REJECTS_TABLE}")

StatementMeta(, 6a0d9aa1-9d23-48cb-bdb9-60cb41efb715, 4, Finished, Available, Finished, False)

Silver batch ID: silver_20260603_005512_8aaf4d60
CSV source:      bronze_citadel_mga.dbo.bronze_agents
API source:      bronze_citadel_mga.dbo.bronze_agents_api
Target:          silver_agents
Rejects:         silver_agents_rejects


In [3]:
# Read both bronze sources and confirm schemas match before attempting UNION.
# If column names or types differ between CSV and API bronze, the UNION will
# either fail or produce nulls in mismatched columns. Better to catch that here.

csv_df = spark.read.table(SOURCE_CSV)
api_df = spark.read.table(SOURCE_API)

print(f"CSV source:  {csv_df.count()} rows, {len(csv_df.columns)} cols")
print(f"API source:  {api_df.count()} rows, {len(api_df.columns)} cols")
print()
print(f"CSV columns: {csv_df.columns}")
print()
print(f"API columns: {api_df.columns}")
print()

# Check for column mismatches between the two sources.
# Domain columns should be identical. Lineage columns will differ
# (_source_file vs _source_endpoint etc) — that's expected and handled below.
csv_domain = set(csv_df.columns) - {"_ingestion_timestamp", "_source_file", "_batch_id", "_ingestion_method"}
api_domain  = set(api_df.columns) - {"_ingestion_timestamp", "_source_endpoint", "_batch_id", "_ingestion_method"}

only_in_csv = csv_domain - api_domain
only_in_api = api_domain - csv_domain

print(f"Domain cols only in CSV: {only_in_csv if only_in_csv else 'none'}")
print(f"Domain cols only in API: {only_in_api if only_in_api else 'none'}")

StatementMeta(, 6a0d9aa1-9d23-48cb-bdb9-60cb41efb715, 5, Finished, Available, Finished, False)

CSV source:  200 rows, 12 cols
API source:  200 rows, 12 cols

CSV columns: ['agent_id', 'first_name', 'last_name', 'agency_name', 'state', 'tier', 'commission_rate', 'hire_date', '_ingestion_timestamp', '_source_file', '_batch_id', '_ingestion_method']

API columns: ['agency_name', 'agent_id', 'commission_rate', 'first_name', 'hire_date', 'last_name', 'state', 'tier', '_ingestion_timestamp', '_source_endpoint', '_batch_id', '_ingestion_method']

Domain cols only in CSV: none
Domain cols only in API: none


In [4]:
# Before UNIONing, standardize the lineage columns so both sources have
# the same shape. We keep _batch_id (renamed to _bronze_batch_id) and
# _ingestion_method from both sources, and add a _source_system column
# so we can tell CSV rows from API rows after the UNION.
# Drop source-specific lineage (_source_file, _source_endpoint,
# _ingestion_timestamp) — silver adds its own timestamp.

csv_standardized = (
    csv_df
    .withColumnRenamed("_batch_id", "_bronze_batch_id")
    .withColumn("_source_system", lit("csv"))
    .drop("_ingestion_timestamp", "_source_file", "_ingestion_method")
)

api_standardized = (
    api_df
    .withColumnRenamed("_batch_id", "_bronze_batch_id")
    .withColumn("_source_system", lit("api"))
    .drop("_ingestion_timestamp", "_source_endpoint", "_ingestion_method")
)

# UNION both sources. We now have up to 400 rows (200 CSV + 200 API)
# before deduplication.
unioned_df = csv_standardized.unionByName(api_standardized)

print(f"CSV rows after standardize:  {csv_standardized.count()}")
print(f"API rows after standardize:  {api_standardized.count()}")
print(f"Combined rows after UNION:   {unioned_df.count()}")
print()
print(f"Columns: {unioned_df.columns}")

StatementMeta(, 6a0d9aa1-9d23-48cb-bdb9-60cb41efb715, 6, Finished, Available, Finished, False)

CSV rows after standardize:  200
API rows after standardize:  200
Combined rows after UNION:   400

Columns: ['agent_id', 'first_name', 'last_name', 'agency_name', 'state', 'tier', 'commission_rate', 'hire_date', '_bronze_batch_id', '_source_system']


In [5]:
# Deduplicate by agent_id. CSV is primary — when both sources have the same
# agent_id, the CSV row survives. API rows only make it through if there's
# no CSV equivalent.
#
# Implementation: assign a priority (CSV=1, API=2), rank rows within each
# agent_id partition by priority, keep only rank=1. Standard pattern for
# source-priority deduplication in PySpark.

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Lower number = higher priority.
priority_df = unioned_df.withColumn(
    "_priority",
    when(col("_source_system") == "csv", 1).otherwise(2)
)

# Within each agent_id, rank rows by priority ascending.
# The CSV row gets rank 1; API row (if present) gets rank 2.
window_spec = Window.partitionBy("agent_id").orderBy("_priority")

ranked_df = priority_df.withColumn("_rank", row_number().over(window_spec))

# Keep only the top-ranked row per agent_id, then drop the helper columns.
deduped_df = (
    ranked_df
    .filter(col("_rank") == 1)
    .drop("_priority", "_rank")
)

# Count how many rows came from each source after deduplication.
csv_count = deduped_df.filter(col("_source_system") == "csv").count()
api_only_count = deduped_df.filter(col("_source_system") == "api").count()

print(f"Rows after deduplication:  {deduped_df.count()}")
print(f"  From CSV:                {csv_count}")
print(f"  From API only:           {api_only_count}")
print()
print(f"Columns: {deduped_df.columns}")

StatementMeta(, 6a0d9aa1-9d23-48cb-bdb9-60cb41efb715, 7, Finished, Available, Finished, False)

Rows after deduplication:  200
  From CSV:                200
  From API only:           0

Columns: ['agent_id', 'first_name', 'last_name', 'agency_name', 'state', 'tier', 'commission_rate', 'hire_date', '_bronze_batch_id', '_source_system']


In [6]:
# Validate deduped rows against three business rules.
# Failing rows go to silver_agents_rejects with a reject_reason column.
# Passing rows go to silver_agents.
#
# reject_reason concatenates all failures for a row so nothing is hidden.
# A row with two failures gets one reason string, not two separate rejects.

reject_reasons = (
    deduped_df
    .withColumn(
        "invalid_tier",
        when(~col("tier").isin(list(VALID_TIERS)), "invalid tier").otherwise(None)
    )
    .withColumn(
        "invalid_rate",
        when(
            (col("commission_rate") < 0.0) | (col("commission_rate") > 0.20),
            "commission_rate out of range"
        ).otherwise(None)
    )
    .withColumn(
        "null_hire_date",
        when(col("hire_date").isNull(), "null hire_date").otherwise(None)
    )
    .withColumn(
        "reject_reason",
        concat_ws(" | ", col("invalid_tier"), col("invalid_rate"), col("null_hire_date"))
    )
    .drop("invalid_tier", "invalid_rate", "null_hire_date")
)

# Split into clean and rejected.
# A row is rejected if reject_reason is non-empty.
rejects_df = reject_reasons.filter(col("reject_reason") != "")
clean_df   = reject_reasons.filter(col("reject_reason") == "")

reject_count = rejects_df.count()
clean_count  = clean_df.count()
total_count  = deduped_df.count()

print(f"Total rows:    {total_count}")
print(f"Clean rows:    {clean_count}")
print(f"Rejected rows: {reject_count}")
print()

assert clean_count + reject_count == total_count, (
    f"Row count mismatch: clean={clean_count} + rejects={reject_count} != total={total_count}"
)
print("Row count assertion passed.")

if reject_count > 0:
    print("\nSample rejects:")
    rejects_df.select("agent_id", "tier", "commission_rate", "hire_date", "reject_reason").show(truncate=False)
else:
    print("No rejects — all rows passed validation.")

StatementMeta(, 6a0d9aa1-9d23-48cb-bdb9-60cb41efb715, 8, Finished, Available, Finished, False)

Total rows:    200
Clean rows:    200
Rejected rows: 0

Row count assertion passed.
No rejects — all rows passed validation.


In [7]:
# Add silver lineage to clean rows, then write both tables.
# Rejects get the same lineage so they're traceable to this silver run.

silver_df = (
    clean_df
    .drop("reject_reason")
    .withColumn("_silver_load_timestamp", current_timestamp())
    .withColumn("_silver_batch_id", lit(SILVER_BATCH_ID))
)

rejects_silver_df = (
    rejects_df
    .withColumn("_silver_load_timestamp", current_timestamp())
    .withColumn("_silver_batch_id", lit(SILVER_BATCH_ID))
)

# Write clean rows.
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_TABLE)
)
print(f"Wrote {silver_df.count()} rows to {TARGET_TABLE}")

# Write rejects. Always write this table even if empty so downstream
# queries and Power BI references don't break on a missing table.
(
    rejects_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(REJECTS_TABLE)
)
print(f"Wrote {rejects_silver_df.count()} rows to {REJECTS_TABLE}")

StatementMeta(, 6a0d9aa1-9d23-48cb-bdb9-60cb41efb715, 9, Finished, Available, Finished, False)

Wrote 200 rows to silver_agents
Wrote 0 rows to silver_agents_rejects


In [10]:
# Read both tables back and confirm the round trip.
# Key check: clean + rejects = total deduped rows. Math must always balance.

verify_clean   = spark.read.table(TARGET_TABLE)
verify_rejects = spark.read.table(REJECTS_TABLE)

clean_count   = verify_clean.count()
reject_count  = verify_rejects.count()
total_deduped = deduped_df.count()

print(f"silver_agents verified:")
print(f"  Clean rows:    {clean_count}")
print(f"  Rejected rows: {reject_count}")
print(f"  Total deduped: {total_deduped}")
print(f"  Math checks:   {clean_count + reject_count} == {total_deduped}: {clean_count + reject_count == total_deduped}")
print()

required_lineage = ["_bronze_batch_id", "_silver_load_timestamp", "_silver_batch_id"]
has_lineage = all(c in verify_clean.columns for c in required_lineage)
print(f"  Lineage cols:  {'present' if has_lineage else 'MISSING'}")
print(f"  Source system breakdown:")

verify_clean.groupBy("_source_system").count().show()

print("Sample:")
verify_clean.select(
    "agent_id", "tier", "commission_rate", "_source_system", "_silver_batch_id"
).show(5, truncate=False)

StatementMeta(, 6a0d9aa1-9d23-48cb-bdb9-60cb41efb715, 12, Finished, Available, Finished, False)

silver_agents verified:
  Clean rows:    200
  Rejected rows: 0
  Total deduped: 200
  Math checks:   200 == 200: True

  Lineage cols:  present
  Source system breakdown:
+--------------+-----+
|_source_system|count|
+--------------+-----+
|           csv|  200|
+--------------+-----+

Sample:
+--------+------+---------------+--------------+-------------------------------+
|agent_id|tier  |commission_rate|_source_system|_silver_batch_id               |
+--------+------+---------------+--------------+-------------------------------+
|AGT-001 |BRONZE|0.08           |csv           |silver_20260603_005512_8aaf4d60|
|AGT-002 |BRONZE|0.08           |csv           |silver_20260603_005512_8aaf4d60|
|AGT-003 |BRONZE|0.08           |csv           |silver_20260603_005512_8aaf4d60|
|AGT-004 |GOLD  |0.12           |csv           |silver_20260603_005512_8aaf4d60|
|AGT-005 |BRONZE|0.08           |csv           |silver_20260603_005512_8aaf4d60|
+--------+------+---------------+--------------+--------